# Part I — 1. Construct RDMs


In [1]:
from pathlib import Path
import torch
from torch.utils.data import DataLoader
from core import SantoroDataset, MODEL_CLASSES, extract_activations, load_yamnet_activations
from rsa_toolbox import compute_rdm

#### 1.1 STG brain data

In [2]:
dataset = SantoroDataset()
brain_responses = dataset.brain_responses.numpy()
brain_rdm = compute_rdm(brain_responses, metric="correlation")
dataloader = DataLoader(dataset, batch_size=16, shuffle=False, num_workers=0)

#### 1.2 Untrained models


In [3]:
untrained_activations, untrained_rdms = {}, {}

for name, model_class in MODEL_CLASSES.items():
    model = model_class(num_classes=50).cpu()
    layers = extract_activations(dataloader, model)
    untrained_activations[name] = layers
    untrained_rdms[name] = {}
    for layer, values in layers.items():
        patterns = values.reshape(len(dataset), -1).numpy()
        untrained_rdms[name][layer] = compute_rdm(patterns, metric="correlation")
    print(f"{name}: {len(layers)} RDMs")

waveform: 6 RDMs
uninspired: 6 RDMs
inspired: 5 RDMs


#### 1.3 Trained models

Load all five supplied runs per architecture and keep their RDMs separate for the comparisons in point 2. No models are trained here.


In [4]:
trained_activations, trained_rdms = {}, {}

for name, model_class in MODEL_CLASSES.items():
    trained_activations[name], trained_rdms[name] = {}, {}
    checkpoints = sorted(Path("models").glob(f"{name}_run*_best.pt"))
    for checkpoint in checkpoints:
        model = model_class(num_classes=50).cpu()
        model.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True))
        layers = extract_activations(dataloader, model)
        run = checkpoint.stem
        trained_activations[name][run] = layers
        trained_rdms[name][run] = {}
        for layer, values in layers.items():
            patterns = values.reshape(len(dataset), -1).numpy()
            trained_rdms[name][run][layer] = compute_rdm(patterns, metric="correlation")
    print(f"{name}: {len(checkpoints)} runs processed")

waveform: 5 runs processed
uninspired: 5 runs processed
inspired: 5 runs processed


#### 1.4 YAMNet

In [5]:
yamnet_activations = load_yamnet_activations()
yamnet_rdms = {}
for layer, patterns in yamnet_activations.items():
    yamnet_rdms[layer] = compute_rdm(patterns, metric="correlation")
print(f"YAMNet: {len(yamnet_rdms)} RDMs")

YAMNet: 15 RDMs
